<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Очищення даних у pandas (Data Cleaning)

**Навіщо це потрібно.** Реальні дані майже ніколи не бувають «чистими»: у них є зайві колонки, дублікати, різний регістр, пропуски, числа, записані словами, дати в різних форматах та різні одиниці виміру. Якщо аналізувати такі дані «як є», середні, групування та графіки будуть неправильними. Тому будь-який аналіз починається з очищення (data cleaning / data wrangling).

**Що ми зробимо на лекції:**

| Крок | Тема | Інструменти pandas |
|------|------|--------------------|
| 0 | Завантаження та первинний огляд | `read_csv`, `info`, `isna` |
| 1 | Колонки: видалення зайвих, перейменування | `drop`, `rename` |
| 2 | Дублікати | `duplicated`, `drop_duplicates` |
| 3 | Текстові та категоріальні дані | `str.strip`, `str.title`, `replace`, `astype('category')` |
| 4 | Числові дані: слова замість чисел, пропуски | `replace`, `to_numeric`, `fillna`, `astype` |
| 5 | Дати в різних форматах | `to_datetime`, `dateutil.parser` |
| 6 | Різні одиниці виміру (кг / фунти) | власна функція + `apply` |
| 7 | Підсумкова перевірка та збереження результату | `info`, `to_csv` |

## Крок 0. Підготовка та первинний огляд даних

Спочатку імпортуємо `pandas` і читаємо CSV-файл прямо з GitHub. Одразу налаштуємо відображення дробових чисел (2 знаки після коми), щоб таблиці були читабельними.

In [2]:
import pandas as pd

# показувати дробові числа з 2 знаками після коми (впливає лише на відображення, не на самі дані)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/cpython-projects/da_27_07_2026/refs/heads/main/lesson_13_data.csv')
df

,Unnamed: 0,ID,Name,gender,Age,Salary ($),Hire Date,Country,Weight
0,0,1,Alice,F,25,50000.00,2023-01-10,USA,70kg
1,1,2,Bob,M,Thirty,60000.00,2022-12-05,U.S.A.,154lbs
2,2,3,CHARLIE,M,40,NaN,01/15/2021,UK,65kg
3,3,4,Dave,M,22,70000.00,"March 1, 2020",United Kingdom,140lbs
4,4,5,Eve,F,NaN,80000.00,2021-07-20,canada,75kg
5,5,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs
6,6,7,George,M,29,NaN,2020-05-30,Germany,80kg
7,7,8,Hannah,F,33,150000.00,"July 4, 2018",France,180lbs
8,8,9,Isaac,M,45,200000.00,06-01-2017,France,90kg
9,9,10,Jack,M,Twenty,250000.00,2016-09-15,Brazil,200lbs


### Які проблеми видно одразу?

Уважно подивимось на таблицю. Це і є наш «план робіт»:

1. **`Unnamed: 0`** — зайва колонка (це старий індекс, який хтось зберіг у файл).
2. **Назви колонок** записані непослідовно: `ID`, `Name`, `gender`, `Salary ($)`, `Hire Date`. Спецсимволи та пробіли в назвах незручні.
3. **Дублікати**: рядки з `Eve` та `Frank` зустрічаються двічі.
4. **Імена**: `CHARLIE` (регістр), `Bob` із зайвими пробілами.
5. **Age**: значення `Thirty`, `Twenty` (слова) та пропуск `NaN`; через це колонка має тип `object`.
6. **Salary**: є пропуски.
7. **Hire Date**: п'ять різних форматів дат, і колонка досі просто текст.
8. **Country**: `USA` / `U.S.A.`, `UK` / `United Kingdom`, `canada` / `Canada`.
9. **Weight**: змішані одиниці (`kg` та `lbs`) і це текст, а не число.

Корисно також одразу подивитись на типи даних і кількість пропусків.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  12 non-null     int64  
 1   ID          12 non-null     int64  
 2   Name        12 non-null     object 
 3   gender      12 non-null     object 
 4   Age         10 non-null     object 
 5   Salary ($)  10 non-null     float64
 6   Hire Date   12 non-null     object 
 7   Country     12 non-null     object 
 8   Weight      12 non-null     object 
dtypes: float64(1), int64(2), object(6)
memory usage: 996.0+ bytes


In [ ]:
# кількість пропущених значень у кожній колонці
df.isna().sum()

> **Що бачимо.** `age` має тип `object`, хоча це має бути число. Пропуски є в `Age` (1) та `Salary ($)` (2). Тип `object` майже завжди означає, що в колонці змішані або текстові значення.

## Крок 1. Колонки: видалення зайвого та перейменування

**Навіщо.** Зайві колонки засмічують таблицю, а зручні назви (маленькі літери, без пробілів і спецсимволів) дозволяють писати `df.age` замість `df['Age']` і уникати помилок.

Спочатку подивимось на список колонок.

In [5]:
df.columns.to_list()

['Unnamed: 0',
 'ID',
 'Name',
 'gender',
 'Age',
 'Salary ($)',
 'Hire Date',
 'Country',
 'Weight']

Колонка `Unnamed: 0` — це старий індекс, який потрапив у файл. Інформації вона не несе, тому видаляємо її.

> 💡 **Порада.** Таку колонку можна було б відкинути ще при читанні: `pd.read_csv(url, index_col=0)`.

In [6]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df

,ID,Name,gender,Age,Salary ($),Hire Date,Country,Weight
0,1,Alice,F,25,50000.00,2023-01-10,USA,70kg
1,2,Bob,M,Thirty,60000.00,2022-12-05,U.S.A.,154lbs
2,3,CHARLIE,M,40,NaN,01/15/2021,UK,65kg
3,4,Dave,M,22,70000.00,"March 1, 2020",United Kingdom,140lbs
4,5,Eve,F,NaN,80000.00,2021-07-20,canada,75kg
5,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs
6,7,George,M,29,NaN,2020-05-30,Germany,80kg
7,8,Hannah,F,33,150000.00,"July 4, 2018",France,180lbs
8,9,Isaac,M,45,200000.00,06-01-2017,France,90kg
9,10,Jack,M,Twenty,250000.00,2016-09-15,Brazil,200lbs


Тепер перейменуємо колонки за єдиним стилем **snake_case** (маленькі літери, слова через `_`). Словник має вигляд `{'стара назва': 'нова назва'}`. Колонку `gender` не чіпаємо: вона вже названа правильно.

In [7]:
df.rename(columns={
    'ID': 'id',
    'Name': 'employee',
    'Age': 'age',
    'Salary ($)': 'employee_salary',
    'Hire Date': 'hire_date',
    'Country': 'country',
    'Weight': 'employee_weight'
}, inplace=True)
df

,id,employee,gender,age,employee_salary,hire_date,country,employee_weight
0,1,Alice,F,25,50000.00,2023-01-10,USA,70kg
1,2,Bob,M,Thirty,60000.00,2022-12-05,U.S.A.,154lbs
2,3,CHARLIE,M,40,NaN,01/15/2021,UK,65kg
3,4,Dave,M,22,70000.00,"March 1, 2020",United Kingdom,140lbs
4,5,Eve,F,NaN,80000.00,2021-07-20,canada,75kg
5,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs
6,7,George,M,29,NaN,2020-05-30,Germany,80kg
7,8,Hannah,F,33,150000.00,"July 4, 2018",France,180lbs
8,9,Isaac,M,45,200000.00,06-01-2017,France,90kg
9,10,Jack,M,Twenty,250000.00,2016-09-15,Brazil,200lbs


## Крок 2. Дублікати

**Навіщо.** Повторені рядки «подвоюють» вплив одних і тих самих спостережень: сума, середнє та кількість будуть викривлені.

`duplicated()` повертає `True` для кожного рядка, який повністю повторює **попередній**. Сума `True` дає кількість дублікатів.

In [8]:
df.duplicated().sum()

np.int64(2)

Перш ніж видаляти, корисно **подивитись**, що саме ми видаляємо. Параметр `keep=False` показує всі копії (і оригінал, і дублікат).

In [9]:
df[df.duplicated(keep=False)]

,id,employee,gender,age,employee_salary,hire_date,country,employee_weight
4,5,Eve,F,NaN,80000.00,2021-07-20,canada,75kg
5,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs
10,5,Eve,F,NaN,80000.00,2021-07-20,canada,75kg
11,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs


Видаляємо дублікати. За замовчуванням залишається **перше** входження. Після видалення в індексі з'являються «дірки» (10 та 11), тому скидаємо індекс через `reset_index(drop=True)`.

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df

## Крок 3. Текстові та категоріальні дані

### 3.1. Імена: пробіли та регістр

**Проблеми:** `CHARLIE` (усі великі літери) та `Bob` із пробілами навколо (їх не видно в таблиці, але вони є, тому `'Bob' != '  Bob  '` і групування чи пошук за іменем можуть «не спрацювати»).

Використовуємо рядкові методи через `.str`:
- `str.strip()` прибирає пробіли по краях,
- `str.title()` робить першу літеру великою, решту маленькими.

In [10]:
print(repr(df.employee[1]))          # видно зайві пробіли
df['employee'] = df.employee.str.strip().str.title()
print(repr(df.employee[1]))          # тепер чисто
df

'  Bob  '
'Bob'


,id,employee,gender,age,employee_salary,hire_date,country,employee_weight
0,1,Alice,F,25,50000.00,2023-01-10,USA,70kg
1,2,Bob,M,Thirty,60000.00,2022-12-05,U.S.A.,154lbs
2,3,Charlie,M,40,NaN,01/15/2021,UK,65kg
3,4,Dave,M,22,70000.00,"March 1, 2020",United Kingdom,140lbs
4,5,Eve,F,NaN,80000.00,2021-07-20,canada,75kg
5,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs
6,7,George,M,29,NaN,2020-05-30,Germany,80kg
7,8,Hannah,F,33,150000.00,"July 4, 2018",France,180lbs
8,9,Isaac,M,45,200000.00,06-01-2017,France,90kg
9,10,Jack,M,Twenty,250000.00,2016-09-15,Brazil,200lbs


### 3.2. Категоріальна колонка `gender`

Якщо в колонці лише кілька повторюваних значень (тут `F` та `M`), доцільно використати тип **`category`**. Він займає менше пам'яті та явно показує, що це категорії, а не довільний текст.

Спочатку перевіримо, які унікальні значення є (чи немає помилок на кшталт `male`, `Male`, `m`).

In [11]:
df.gender.unique()

array(['F', 'M'], dtype=object)

In [12]:
df['gender'] = df.gender.astype('category')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               12 non-null     int64   
 1   employee         12 non-null     object  
 2   gender           12 non-null     category
 3   age              10 non-null     object  
 4   employee_salary  10 non-null     float64 
 5   hire_date        12 non-null     object  
 6   country          12 non-null     object  
 7   employee_weight  12 non-null     object  
dtypes: category(1), float64(1), int64(1), object(5)
memory usage: 940.0+ bytes


### 3.3. Країни: різні написання однієї й тієї самої країни

Подивимось на унікальні значення.

In [13]:
df.country.unique()

array(['USA', 'U.S.A.', 'UK', 'United Kingdom', 'canada', 'Canada',
       'Germany', 'France', 'Brazil'], dtype=object)

Ми бачимо `canada` / `Canada`, `USA` / `U.S.A.`, `UK` / `United Kingdom`. Для комп'ютера це різні країни.

Виправляємо у два кроки:
1. Приводимо регістр до єдиного вигляду (`title()`); після цього `canada` стає `Canada`, а `USA` перетворюється на `Usa`.
2. Замінюємо синоніми за допомогою словника в `replace()`. Це правильний підхід, коли варіантів небагато. Якщо їх багато, краще використовувати довідник країн.

In [14]:
df['country'] = df.country.str.title()
df.country.unique()

array(['Usa', 'U.S.A.', 'Uk', 'United Kingdom', 'Canada', 'Germany',
       'France', 'Brazil'], dtype=object)

In [15]:
df['country'] = df.country.replace({
    'Usa': 'U.S.A.',
    'Uk': 'United Kingdom',
})
df.country.value_counts()

,count
country,
Canada,4
U.S.A.,2
United Kingdom,2
France,2
Germany,1
Brazil,1


`value_counts()` дозволяє швидко переконатись, що лишилось 6 акуратних значень (без дублів на кшталт `Usa` та `U.S.A.`).

## Крок 4. Числові дані

### 4.1. Колонка `age`: слова замість чисел і пропуски

Спочатку подивимось, які значення в колонці.

In [16]:
df.age.unique()

array(['25', 'Thirty', '40', '22', nan, '35', '29', '33', '45', 'Twenty'],
      dtype=object)

Значення `Thirty` та `Twenty` — це числа, записані словами, через них колонка має тип `object`. Замінимо їх на цифри (поки що залишаємо рядками, як і решта значень).

In [17]:
df['age'] = df.age.replace({
    'Thirty': '30',
    'Twenty': '20'
})

Спробуємо одразу перетворити колонку на ціле число. **Це не спрацює**, і це добре видно на практиці: у колонці є `NaN`, а тип `int` у pandas не може зберігати пропуски.

In [18]:
try:
    df['age'] = df.age.astype('int')
except ValueError as e:
    print('Помилка:', e)

Помилка: cannot convert float NaN to integer


Правильний порядок дій:

1. `pd.to_numeric(..., errors='coerce')` перетворює текст на числа; усе, що не вдалося розпізнати, стає `NaN` (`errors='coerce'` — це «не падай, а став NaN»). Колонка стає типом `float`.
2. Заповнюємо пропуски за допомогою `fillna(...)`.
3. Тільки після цього безпечно переходимо до `int`.

Пропуск ми заповнюємо **середнім віком**. Це найпростіший варіант, але треба розуміти, що ми «вигадуємо» значення: для аналізу віку краще, якщо пропусків мало, або вказати про це в звіті. Альтернативи: медіана, значення за групою (наприклад, за країною) або залишити `NaN`.

In [19]:
df['age'] = pd.to_numeric(df.age, errors='coerce')
df

,id,employee,gender,age,employee_salary,hire_date,country,employee_weight
0,1,Alice,F,25.00,50000.00,2023-01-10,U.S.A.,70kg
1,2,Bob,M,30.00,60000.00,2022-12-05,U.S.A.,154lbs
2,3,Charlie,M,40.00,NaN,01/15/2021,United Kingdom,65kg
3,4,Dave,M,22.00,70000.00,"March 1, 2020",United Kingdom,140lbs
4,5,Eve,F,NaN,80000.00,2021-07-20,Canada,75kg
5,6,Frank,M,35.00,120000.00,08/10/2019,Canada,165lbs
6,7,George,M,29.00,NaN,2020-05-30,Germany,80kg
7,8,Hannah,F,33.00,150000.00,"July 4, 2018",France,180lbs
8,9,Isaac,M,45.00,200000.00,06-01-2017,France,90kg
9,10,Jack,M,20.00,250000.00,2016-09-15,Brazil,200lbs


In [20]:
df['age'] = df.age.fillna(df.age.mean())   # середнє = 31.44..., у int стане 31
df['age'] = df.age.astype('int')
df

,id,employee,gender,age,employee_salary,hire_date,country,employee_weight
0,1,Alice,F,25,50000.00,2023-01-10,U.S.A.,70kg
1,2,Bob,M,30,60000.00,2022-12-05,U.S.A.,154lbs
2,3,Charlie,M,40,NaN,01/15/2021,United Kingdom,65kg
3,4,Dave,M,22,70000.00,"March 1, 2020",United Kingdom,140lbs
4,5,Eve,F,31,80000.00,2021-07-20,Canada,75kg
5,6,Frank,M,35,120000.00,08/10/2019,Canada,165lbs
6,7,George,M,29,NaN,2020-05-30,Germany,80kg
7,8,Hannah,F,33,150000.00,"July 4, 2018",France,180lbs
8,9,Isaac,M,45,200000.00,06-01-2017,France,90kg
9,10,Jack,M,20,250000.00,2016-09-15,Brazil,200lbs


### 4.2. Колонка `employee_salary`: пропуски

У двох працівників (`Charlie`, `George`) зарплата не вказана. Тут ми **свідомо не заповнюємо пропуски**: вигадувати чужу зарплату (наприклад, середньою) було б помилкою для більшості задач.

Важливо знати: методи `mean()`, `median()`, `sum()` за замовчуванням **пропускають** `NaN`, тому статистика все одно порахується (по 8 значеннях).

In [21]:
print('Пропусків у зарплаті:', df.employee_salary.isna().sum())
print('Середня зарплата (NaN пропускаються):', round(df.employee_salary.mean(), 2))
print('Медіана:', df.employee_salary.median())

# Якщо для задачі потрібні заповнені значення, зазвичай беруть медіану (вона стійка до викидів):
# df['employee_salary'] = df.employee_salary.fillna(df.employee_salary.median())

Пропусків у зарплаті: 2
Середня зарплата (NaN пропускаються): 118000.0
Медіана: 100000.0


## Крок 5. Дати в різних форматах

**Навіщо.** Поки дата є текстом, з нею не можна рахувати (стаж, різницю між датами), сортувати за часом чи групувати за роками.

У колонці `hire_date` змішані формати: `2023-01-10`, `01/15/2021`, `March 1, 2020`, `July 4, 2018`, `06-01-2017`.

### Спроба 1: `pd.to_datetime(..., errors='coerce')`

Створимо тестову колонку `new_date` і подивимось, що вийде.

In [ ]:
df['new_date'] = pd.to_datetime(df.hire_date, errors='coerce')
df[['hire_date', 'new_date']]

**Результат поганий:** для половини рядків з'явились `NaT` (Not a Time — «пропуск» для дат). `to_datetime` вибирає формат за першими значеннями і вважає рядки в інших форматах помилковими, а `errors='coerce'` мовчки замінює їх на `NaT`. Тобто **половина дат втрачена**, і це той випадок, коли `coerce` небезпечний: завжди перевіряйте, скільки `NaT` вийшло.

### Спроба 2: `dateutil.parser.parse`

Бібліотека `dateutil` вміє «здогадуватись» про формат кожного значення окремо. Застосуємо її до кожного рядка через `apply`.

In [ ]:
from dateutil import parser

df['hire_date'] = df.hire_date.apply(parser.parse)
df[['hire_date', 'new_date']]

Тепер усі 10 дат розпізнано. Тестова колонка `new_date` більше не потрібна, тому видаляємо її, щоб не залишати в таблиці сміття.

> ⚠️ **Пастка неоднозначних дат.** Запис `06-01-2017` може означати і 1 червня, і 6 січня. За замовчуванням `parser.parse` вважає, що спочатку йде **місяць** (американський формат), тобто отримали 1 червня 2017. Якщо джерело даних європейське, потрібно вказувати `dayfirst=True`. Завжди уточнюйте формат у власника даних.
>
> 💡 У pandas 2.0+ є вбудований варіант: `pd.to_datetime(df.hire_date, format='mixed')`. Він працює без `dateutil`.

In [ ]:
df.drop(columns=['new_date'], inplace=True)
df.info()

## Крок 6. Різні одиниці виміру: вага в кг та фунтах

**Проблема.** У колонці `employee_weight` записано і `70kg`, і `154lbs`. Це текст, і до того ж різні одиниці, тому порівнювати їх не можна.

**План:** написати функцію, яка приймає один рядок, визначає одиницю та повертає вагу в кілограмах (число), а потім застосувати її до всієї колонки через `apply`.

Коефіцієнт переведення: **1 фунт = 0.45359237 кг**.

In [ ]:
LBS_TO_KG = 0.45359237


def convert_weight(item):
    item = item.lower().strip()
    if 'kg' in item:
        return float(item.replace('kg', ''))

    if 'lbs' in item:
        return float(item.replace('lbs', '')) * LBS_TO_KG

    return float(item)   # якщо одиниці немає, вважаємо, що це вже кг


df['weight_kg'] = df.employee_weight.apply(convert_weight).round(2)
df[['employee', 'employee_weight', 'weight_kg']]

Переконуємось, що результат правильний: `154lbs` ≈ 69.85 кг, `200lbs` ≈ 90.72 кг. Початкову текстову колонку можна прибрати (у реальному проєкті її часто залишають до кінця перевірки).

In [ ]:
df.drop(columns=['employee_weight'], inplace=True)
df

## Крок 7. Підсумкова перевірка та збереження

Після очищення завжди робимо контрольний огляд:
- правильні типи (`int`, `float`, `datetime64`, `category`),
- немає дублікатів,
- пропуски лише там, де ми їх свідомо залишили.

In [ ]:
df.info()

In [ ]:
print('Дублікатів:', df.duplicated().sum())
print(df.isna().sum())

Зберігаємо очищені дані в новий файл. `index=False` не дає pandas знову записати індекс у файл, з якого ми на початку прибирали `Unnamed: 0`.

In [ ]:
df.to_csv('lesson_12_clean.csv', index=False)
df

**Золоте правило очищення:** після кожного кроку перевіряйте результат (`unique()`, `info()`, `value_counts()`), ніколи не довіряйте «мовчазним» операціям на кшталт `errors='coerce'`.